# Make Models

B Steele, Colorado State University, ROSSyndicate — last updated Sep 19, 2026

**Pipeline step 04 · Python · run from `regional_clarity/`**

- **Purpose:** For each of 10 seeds (601–610): correlation pruning, backward elimination, gap-aware tuning, and 4-fold XGBoost training (40 models in all).
- **Inputs:** `aquamatch_files/base_with_splits.parquet`.
- **Outputs:** `xg_models/v3_production/seed{601–610}/`: `backward_elim_summary.json`, `final_tune_xgboost.json`, `xgboost_fold{1–4}.json`, `final_eval_summary.json`, `holdout_predictions.parquet`.
- **Previous:** [03 · Split data](03_split_data.ipynb)
- **Next:** [05 · Evaluate ensemble](05_evaluate_ensemble.ipynb)

This notebook trains the production Secchi-depth model: an ensemble of XGBoost models over ten independent HUC8-random CV arrangements built in `03_split_data.ipynb`, evaluated on that notebook's one fixed, never-trained-on holdout.

**Single algorithm, ten seeds.** An earlier version of this pipeline ensembled 5 seeds × 2 algorithms (XGBoost + LightGBM), for 40 models total. A variance decomposition of that ensemble's holdout predictions found the two algorithms agreed very closely with one another (0.998 correlation between their respective 5-seed ensemble means) while different random seeds disagreed with each other substantially more (~0.99 pairwise correlation); seed choice accounted for roughly 65% of the ensemble's internal prediction spread, versus roughly 13% for model architecture. Given that seed, not algorithm, was the dominant source of useful ensemble variability, this pipeline runs a single algorithm (XGBoost) over 10 independent random seeds instead (`601`–`610`) — 10 seeds × 4 folds = 40 models, the same total model count as the original two-algorithm design, for no measurable accuracy cost and a simpler codebase to maintain.

Each ensemble seed runs its own complete feature-selection + tuning + training pipeline, entirely independently, on its own 4-fold split of the shared CV pool. The final production prediction is the average across all 10 seeds × 4 folds.

## Setup

Import the shared modeling library in `python/`: backward elimination, feature engineering, metrics, the XGBoost wrapper, spatial CV, and gap-aware tuning. Then set the run parameters:

- the 10 ensemble seeds (601–610);
- the number of random-search trials for the initial and final tuning passes (25 and 40);
- the backward-elimination tolerance and seeds;
- the site characteristics and the two curated weather features (`precip_mm_prev30`, `tmax_degC_prev30`) added to the optical candidate pool.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("python").resolve()))
import backward_elim as be
import features
import metrics as M
import model_xgb
import spatial_cv
from tuning import gap_aware_tune

be.register_model("xgboost", model_xgb)

AQUAMATCH_DIR = Path("aquamatch_files")
MODELS_DIR = Path("xg_models") / "v3_production"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "harmonized_value"
MODEL_NAME = "xgboost"
MOD = model_xgb
SEEDS = list(range(601, 611))
N_TRIALS_INITIAL = 25
N_TRIALS_FINAL = 40
ELIM_TOL = 0.005
ELIM_SEED = 47
TUNE_SEED = 47

SITE_COLS = ["elevation_m", "catchment_area_sqkm", "pct_impervious_2006", "pct_urban_2006",
             "pct_forest_2006", "pct_cropland_2006", "pct_wetland_2006", "shore_flag"]
WEATHER_COLS_CURATED = ["precip_mm_prev30", "tmax_degC_prev30"]


def log(seed, msg):
    print(f"[{time.strftime('%H:%M:%S')}][seed{seed}] {msg}", flush=True)

## Load data and build 10 fresh CV-fold arrangements

Reads `03_split_data.ipynb`'s output (the same fixed holdout used everywhere else in this project) and adds 10 new `cvfold_seed{601-610}` columns via `spatial_cv.build_splits` — the holdout itself is rebuilt deterministically from the same `holdout_seed=12`, so it comes back identical to `03`'s; only the CV-pool fold arrangement is new. `03`'s legacy `cvfold_seed{501-505}` columns are left untouched alongside these (unused by this pipeline).

`features.coarsen_site_features()` reduces both the *extremity* and the *precision* of `catchment_area_sqkm` (first winsorized at the training-data 99th percentile, then log-scale binned) and rounds the `pct_*_2006` land-cover fractions to the nearest percentage point, so a heavily-resampled or catchment-outlier waterbody's static, per-comid LakeCat values can't act as a de facto waterbody ID for the model to key a memorized SDD value off of. Log-binning alone does this for the ~560 single-HUC8 waterbodies but not for a waterbody with no real neighbors in the distribution - Lake Powell's catchment area (99.5th percentile among waterbodies) stayed cleanly separated from every other large reservoir even after log-binning, and SHAP analysis confirmed the model was leaning on it there specifically. Winsorizing first forces Powell to collide with its nearest real peers (Great Salt Lake, Lake Fort Peck, Flathead Lake, Utah Lake), which measurably reduced that reliance in practice.

In [2]:
df = pd.read_parquet(AQUAMATCH_DIR / "base_with_splits.parquet")
df = features.add_spectral_indices(df)
df = features.coarsen_site_features(df)
df = spatial_cv.build_splits(df, ensemble_seeds=SEEDS)
print(f"{len(df):,} rows, {df.shape[1]} columns; holdout n={df['is_holdout'].sum():,}")

11,143 rows, 72 columns; holdout n=2,206


## Per-seed pipeline (single algorithm)

`run_seed_pipeline()` builds one seed's models from start to finish and saves every artifact to `xg_models/v3_production/seed{seed}/`. All selection decisions use out-of-fold CV only; the fixed holdout is scored only at the end.

1. **Folds.** Build this seed's 4-fold HUC8 arrangement of the CV pool.
2. **Candidate pool and correlation pruning.** Combine the optical features (harmonized bands and spectral indices) with the site characteristics and the curated weather features. Within each cluster of near-duplicate features (|r| > 0.95), keep the one most correlated with SDD.
3. **Initial gap-aware tuning.** Run a 25-trial random hyperparameter search. Among the trials whose mean validation RMSE is within 2% of the best, choose the one with the smallest train–validation gap. These hyperparameters are held fixed during elimination. Each seed uses its own search seed (`TUNE_SEED + seed`), so the 10 seeds explore different parts of hyperparameter space.
4. **Backward elimination with shadow decoys.** Each real feature gets a row-permuted "shadow" copy that has no relationship to SDD. At each step, drop the weakest real feature if its permutation importance does not beat the shadows. Stop when every remaining feature beats the shadows, or when mean CV RMSE degrades by more than `ELIM_TOL` relative to the best point on the path. The survivors form the seed's final feature set.
5. **Final gap-aware tuning.** Re-run the tuning procedure on the final feature set with 40 trials.
6. **Final training and evaluation.** Train the four fold models, then score them on their own validation folds (CV) and as a 4-model average on the fixed holdout. Save the fold models, the feature and hyperparameter configurations, the metrics, and the holdout predictions.

In [3]:
def run_seed_pipeline(seed, df):
    out_dir = MODELS_DIR / f"seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    t_start = time.time()

    folds = spatial_cv.build_folds(df, seed)
    test = df[df["is_holdout"]].reset_index(drop=True)
    log(seed, f"folds: {[(f.part, len(f.train), len(f.val)) for f in folds]}, holdout n={len(test)}")

    # --- candidate pool + correlation pruning (this seed's own train_val pool) ---
    metadata_cols = {"HUC8", "holdout_part", "is_holdout"} | {f"cvfold_seed{s}" for s in SEEDS}
    extra_cols = set(SITE_COLS) | set(WEATHER_COLS_CURATED) | metadata_cols
    optical_feats = [f for f in features.candidate_feature_list(df) if f not in extra_cols]
    candidates = optical_feats + SITE_COLS + WEATHER_COLS_CURATED

    train_val = pd.concat([f.train for f in folds] + [folds[0].val], ignore_index=True).drop_duplicates(
        subset=["siteSR_id", "date"])
    pruned = features.correlation_prune(train_val, candidates, target=TARGET)
    log(seed, f"candidate pool: {len(candidates)} -> {len(pruned)} after correlation pruning")

    # tuning seed is derived per-ensemble-seed (rather than a single fixed value shared
    # by every seed) so each seed's random search explores a genuinely different draw of
    # hyperparameter combinations - previously every seed reused seed=47 verbatim, so all
    # 10 seeds evaluated the identical set of combinations against their own data instead
    # of diversifying search coverage across the ensemble the way the CV folds and feature
    # selection already do
    seed_tune_seed = TUNE_SEED + seed

    # --- stage 1: initial gap-aware tuning (fixed params for elimination) ---
    log(seed, f"=== initial gap-aware tuning on {len(pruned)}-feature pool ===")
    init_result = gap_aware_tune(MOD, folds, pruned, TARGET, n_trials=N_TRIALS_INITIAL, seed=seed_tune_seed)
    fixed_params = init_result["best_params"]
    log(seed, f"val_rmse={init_result['best_score']:.4f} gap={init_result['best_gap']:.4f}")

    # --- stage 2: backward elimination (single algorithm, no joint step) ---
    log(seed, "=== backward elimination ===")
    elim_result = be.backward_eliminate(MODEL_NAME, folds, TARGET, fixed_params, pruned,
                                        weight_fn=None, tol=ELIM_TOL, seed=ELIM_SEED,
                                        log=lambda m: log(seed, m))
    final_feats = elim_result["final_features"]
    log(seed, f"final ({len(final_feats)}): {final_feats}")

    with open(out_dir / "backward_elim_summary.json", "w") as fh:
        json.dump(dict(starting_candidates=candidates, correlation_pruned=pruned,
                       final_features=final_feats, path=elim_result["path"]), fh, indent=2, default=str)

    # --- stage 3: final gap-aware tuning on the final feature set ---
    log(seed, f"=== final gap-aware tuning on {len(final_feats)}-feature set ===")
    tune_result = gap_aware_tune(MOD, folds, final_feats, TARGET, n_trials=N_TRIALS_FINAL, seed=seed_tune_seed)
    log(seed, f"chosen: val_rmse={tune_result['best_score']:.4f} gap={tune_result['best_gap']:.4f}")
    with open(out_dir / f"final_tune_{MODEL_NAME}.json", "w") as fh:
        json.dump({"model": MODEL_NAME, "features": final_feats, **tune_result}, fh, indent=2, default=str)

    # --- stage 4: final unweighted training + evaluation ---
    params = tune_result["best_params"]
    fold_models = MOD.train_fold_models(folds, final_feats, TARGET, params)
    for i, fm in enumerate(fold_models):
        fm.save_model(str(out_dir / f"{MODEL_NAME}_fold{i+1}.json"))

    cv_preds, cv_ys = [], []
    for f, model in zip(folds, fold_models):
        cv_preds.append(MOD.predict_ensemble([model], f.val[final_feats]))
        cv_ys.append(f.val[TARGET].values)
    cv_metrics = M.all_metrics(np.concatenate(cv_ys), np.concatenate(cv_preds))

    test_pred = MOD.predict_ensemble(fold_models, test[final_feats])
    test_metrics = M.all_metrics(test[TARGET].values, test_pred)
    log(seed, f"FINAL: cv_rmse={cv_metrics['rmse']:.4f} holdout_rmse={test_metrics['rmse']:.4f}")

    summary = {MODEL_NAME: dict(features=final_feats, params=params, cv_metrics=cv_metrics, test_metrics=test_metrics)}
    with open(out_dir / "final_eval_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)

    r = test[["siteSR_id", "date", "HUC4", "HUC8"]].copy()
    r["seed"] = seed
    r["model"] = MODEL_NAME
    r["y"] = test[TARGET].values
    r["pred"] = test_pred
    r.to_parquet(out_dir / "holdout_predictions.parquet")

    log(seed, f"SEED {seed} DONE in {time.time()-t_start:.0f}s")
    return summary

Run the pipeline for all 10 seeds. Each seed is independent, so the CV arrangement, selected features, and hyperparameters can all differ from seed to seed.

In [4]:
all_summaries = {}
for seed in SEEDS:
    all_summaries[seed] = run_seed_pipeline(seed, df)

[16:55:02][seed601] folds: [(1, 6764, 2173), (2, 6551, 2386), (3, 6822, 2115), (4, 6674, 2263)], holdout n=2206


[16:55:03][seed601] candidate pool: 45 -> 34 after correlation pruning


[16:55:03][seed601] === initial gap-aware tuning on 34-feature pool ===


[16:59:20][seed601] val_rmse=1.5008 gap=0.5374


[16:59:20][seed601] === backward elimination ===


...

[19:34:41][seed610] final (33): ['atm_corr_LaSRC', 'green_corr7', 'nir_corr7', 'swir1_corr7', 'temp_corr7', 'precip_mm_prev1', 'tmean_degC_prev7', 'srad_Wm2_prev1', 'precip_mm_prev3', 'srad_Wm2_prev7', 'precip_mm_prev7', 'tmax_degC_prev7', 'tmin_degC_prev7', 'tmean_degC_prev30', 'tmin_degC_prev30', 'srad_Wm2_prev30', 'BR', 'BG', 'NR', 'fai', 'NDVI', 'NDSSI', 'NDWI', 'MNDWI', 'catchment_area_sqkm', 'pct_impervious_2006', 'pct_urban_2006', 'pct_forest_2006', 'pct_cropland_2006', 'pct_wetland_2006', 'shore_flag', 'precip_mm_prev30', 'tmax_degC_prev30']


[19:34:41][seed610] === final gap-aware tuning on 33-feature set ===


[19:40:41][seed610] chosen: val_rmse=1.4683 gap=0.4516


[19:40:58][seed610] FINAL: cv_rmse=1.4701 holdout_rmse=1.2795


[19:40:58][seed610] SEED 610 DONE in 627s


## Per-seed summary

One row per seed: the size of its final feature set, its out-of-fold CV RMSE, and its own 4-model holdout RMSE and bias. Seeds that differ widely here would suggest the result depends heavily on the CV arrangement.

In [5]:
rows = []
for seed, summary in all_summaries.items():
    s = summary[MODEL_NAME]
    rows.append(dict(seed=seed, n_features=len(s["features"]), cv_rmse=s["cv_metrics"]["rmse"],
                     holdout_rmse=s["test_metrics"]["rmse"], holdout_bias=s["test_metrics"]["bias"]))
pd.DataFrame(rows).sort_values("seed")

,seed,n_features,cv_rmse,holdout_rmse,holdout_bias
0,601,26,1.467190,1.270913,0.181762
1,602,26,1.469815,1.266743,0.139566
2,603,33,1.528273,1.270425,0.162404
3,604,29,1.434455,1.272325,0.190365
4,605,26,1.438644,1.271244,0.192054
5,606,28,1.473498,1.262187,0.138934
6,607,28,1.456559,1.276885,0.160003
7,608,29,1.460105,1.266200,0.184919
8,609,30,1.483075,1.282485,0.200381
9,610,33,1.470124,1.279548,0.195797


## Assemble the 10-seed ensemble

40 models (10 seeds x 4 folds). Full evaluation (HUC8 breakdown, seed-vs-seed spread, SHAP) happens in `05_evaluate_ensemble.ipynb`; this is a quick sanity check on the assembled holdout metrics before moving on.

In [6]:
rows = []
for seed in SEEDS:
    rows.append(pd.read_parquet(MODELS_DIR / f"seed{seed}" / "holdout_predictions.parquet"))
all_preds = pd.concat(rows, ignore_index=True)

ensemble_pred = all_preds.groupby(["siteSR_id", "date"]).agg(y=("y", "first"), pred=("pred", "mean")).reset_index()
ensemble_metrics = M.all_metrics(ensemble_pred["y"], ensemble_pred["pred"])
print(f"production ensemble (10 seeds x xgboost, 40 models): {ensemble_metrics}")

production ensemble (10 seeds x xgboost, 40 models): {'rmse': 1.2589116475492872, 'mae': 0.9395769457867804, 'mape': 0.41458014203268273, 'bias': 0.1476610320126887, 'p_bias': 4.577920313906626, 'smape': 0.31761844424437347, 'r2': 0.6264662907389484, 'n': 1833}


## Summary

The production model is a 40-member ensemble — 10 independent HUC8-random CV arrangements x 4 folds each — trained unweighted, with gap-aware hyperparameter selection at every tuning stage and per-seed feature sets chosen by correlation pruning followed by shadow-decoy backward elimination. Every member is evaluated on one fixed HUC8-random holdout that no model ever trains on. Artifacts (fold models, per-seed feature/hyperparameter configs, holdout predictions) are saved under `xg_models/v3_production/seed{601–610}/`. See `05_evaluate_ensemble.ipynb` for the full evaluation (HUC8 breakdown, seed-vs-seed spread, SHAP).